# Lab: ผู้ช่วย AI คัดกรองข้อความสแกม
### Thailand Cyber & Digital Innovation Challenge — Online Workshop
**Claude บน Microsoft Foundry (Azure)**

---

### คุณจะได้อะไรจาก notebook นี้
ผู้ช่วย AI ที่อ่านข้อความภาษาไทย แล้วบอกว่า **สแกม / ปกติ / ข้อมูลบิดเบือน** พร้อมเหตุผล
และดึงของสำคัญออกมา เช่น ลิงก์ เบอร์โทร เลขบัญชี

### เดิน 3 ขั้น
| ขั้น | เวลา | ทำอะไร | ใครทำ |
|---|---|---|---|
| **1** | 15 นาที | รันของที่ให้มา ให้เห็นผลลัพธ์แรก | **ทุกคน** |
| **2** | 15 นาที | แก้คำสั่ง ใส่ข้อความของตัวเอง | **ทุกคน** |
| **3** | 15 นาที | สายโค้ด → ต่อ tool use · สายไม่โค้ด → ออกแบบ flow | เลือกทาง |

> ### ⚠️ เรื่อง key
> key ที่แจกเป็นของกลางรายทีม **มีเพดานค่าใช้จ่าย และจะถูกยกเลิกเย็นวันนี้**
> อย่าเอาไปใส่ในโค้ดที่ push ขึ้น GitHub และอย่าแชร์ออกนอกทีม

> ### วิธีเปิดไฟล์นี้
> ในหน้า repository ที่ได้รับ กดปุ่ม **Code** แล้วเลือก **Codespaces → Create codespace on main**
> ระบบจะเปิด VS Code บนเบราว์เซอร์พร้อม Python ให้อัตโนมัติ ไม่ต้องติดตั้งอะไรลงเครื่อง
> จากนั้นเปิดไฟล์นี้ แล้วมุมขวาบนกด **Select Kernel → Python Environments... → Python 3.11** (`/usr/local/bin/python`)
> อย่าเลือก *Create Python Environment* ที่ระบบไฮไลต์ไว้ หรือ Python 3.13 เพราะไม่มีไลบรารีของแล็บ แล้วกดรันทีละเซลล์ได้เลย

---
## ขั้น 1 — ให้มันทำงานก่อน

In [ ]:
# รันเซลล์นี้ก่อน (ใน Codespaces ของทีม ติดตั้งไว้ให้แล้ว จะเสร็จเกือบทันที)
!pip install -q anthropic

### ใส่ key ของทีม
แก้ 2 บรรทัดข้างล่างให้เป็นค่าที่ได้รับ แล้วรัน

In [ ]:
FOUNDRY_API_KEY  = "วาง-key-ของทีม-ตรงนี้"
FOUNDRY_RESOURCE = "วาง-ชื่อ-resource-ตรงนี้"

from anthropic import AnthropicFoundry

client = AnthropicFoundry(api_key=FOUNDRY_API_KEY, resource=FOUNDRY_RESOURCE)

# โมเดลสองตัวที่เราจะใช้ — ราคาต่างกันเกือบ 40 เท่า
MODEL_FAST  = "claude-haiku-5-5"   # ถูกและเร็ว ใช้คัดกรองทุกข้อความ
MODEL_SMART = "claude-opus-5-5"    # ฉลาดกว่า แพงกว่า ใช้เฉพาะเคสที่ต้องคิด

# ยิงทดสอบ 1 ครั้ง
r = client.messages.create(
    model=MODEL_FAST,
    max_tokens=200,
    messages=[{"role": "user", "content": "ตอบกลับสั้น ๆ ว่า พร้อมแล้ว"}],
)
print("".join(b.text for b in r.content if b.type == "text"))
print("\n✅ ต่อ Azure ได้แล้ว ไปขั้นต่อไปได้เลย")

**ถ้าเห็น ✅ แปลว่าผ่านขั้นที่ยากที่สุดแล้ว**

ติดตรงนี้? วางข้อความ error ทั้งบรรทัดลงแชต Teams วิทยากรจะกวาดตอบรวดเดียวที่จุดตรวจ

---
### ชุดข้อความตัวอย่าง
ข้อความทั้งหมดนี้ **แต่งขึ้นเพื่อการสอน** เบอร์ ลิงก์ และเลขบัญชีเป็นของปลอมทั้งหมด

ค่าที่สองในแต่ละบรรทัดคือ **เฉลย** เอาไว้วัดความแม่นในขั้นท้าย

In [ ]:
SAMPLE_MESSAGES = [
    ("พัสดุของท่านตกค้างที่ศุลกากร กรุณาชำระค่าธรรมเนียม 128 บาท ภายใน 24 ชม. ที่ th-parcel-pay.example", "สแกม"),
    ("คุณได้รับสิทธิ์เงินกู้ฉุกเฉิน 50,000 บาท ดอกเบี้ย 0% ไม่เช็คเครดิต สมัครเลย add LINE: @fastloan-x", "สแกม"),
    ("ธนาคารแจ้ง บัญชีของท่านถูกระงับชั่วคราว ยืนยันตัวตนภายในวันนี้ที่ bank-verify-th.example เพื่อปลดล็อก", "สแกม"),
    ("ที่นี่กองปราบปราม บัญชีคุณพัวพันคดีฟอกเงิน ติดต่อกลับด่วน 02-000-0000 มิฉะนั้นจะออกหมายจับ", "สแกม"),
    ("รับสมัครงานพาร์ทไทม์ กดไลก์คลิปละ 50 บาท โอนไว ทักแชทรับงานเลย", "สแกม"),
    ("กลุ่มลงทุนทองคำ การันตีกำไร 15% ต่อเดือน รับประกันทุน รีบสมัคร เหลือ 3 ที่", "สแกม"),
    ("การไฟฟ้าแจ้งค้างชำระ 1,240 บาท ตัดไฟคืนนี้ ชำระที่ power-bill-th.example ภายใน 2 ชม.", "สแกม"),
    ("แม่จ๊ะ นี่ลูกนะ เปลี่ยนเบอร์ใหม่ พอดีมีเรื่องด่วน โอนเงินให้หน่อย 8,000 บัญชี XXX-X-XXXXX-X", "สแกม"),
    ("ยินดีด้วย! คุณคือผู้โชคดีได้รับโทรศัพท์รุ่นล่าสุด กรอกข้อมูลรับของที่ win-prize-th.example", "สแกม"),
    ("แอปกู้เงินด่วน อนุมัติใน 5 นาที ไม่ต้องค้ำ โหลดได้ที่ลิงก์นี้ ไม่ต้องผ่านสโตร์", "สแกม"),

    ("รหัส OTP ของคุณคือ 884213 ใช้ได้ 5 นาที ธนาคารไม่มีนโยบายขอรหัสนี้จากคุณ", "ปกติ"),
    ("บัญชี xxx-123 มีเงินเข้า 2,500.00 บาท คงเหลือ 18,430.25 บาท เวลา 14:02", "ปกติ"),
    ("แจ้งเตือน นัดหมายพบแพทย์ พรุ่งนี้ 10:30 น. แผนกอายุรกรรม กรุณามาก่อนเวลา 15 นาที", "ปกติ"),
    ("พัสดุเลขที่ TH00123456 ถึงจุดคัดแยกแล้ว คาดว่าจัดส่งพรุ่งนี้ ตรวจสอบสถานะในแอป", "ปกติ"),
    ("โปรโมชันสมาชิก ลด 20% เมื่อซื้อครบ 500 บาท ที่ร้านสาขาใดก็ได้ ถึงสิ้นเดือนนี้", "ปกติ"),
    ("มหาวิทยาลัยแจ้ง เปิดลงทะเบียนเรียนภาคปลาย 1-15 ธ.ค. ผ่านระบบทะเบียนของมหาวิทยาลัย", "ปกติ"),

    ("ด่วน! น้ำมะนาวผสมโซดาอุ่น ฆ่าเซลล์มะเร็งได้ 100% หมอไม่อยากให้คุณรู้ แชร์ต่อด่วน", "ข้อมูลบิดเบือน"),
    ("เขื่อนแตกแล้ว! น้ำจะถึงตัวเมืองใน 2 ชม. ทางการปิดข่าว รีบอพยพ ส่งต่อให้คนที่คุณรัก", "ข้อมูลบิดเบือน"),
    ("ยืนยันแล้ว รัฐจะเก็บภาษีเงินฝากทุกบัญชี 3% เริ่มเดือนหน้า รีบถอนก่อนโดน", "ข้อมูลบิดเบือน"),
    ("คลิปนี้พิสูจน์ว่าคนดังพูดแบบนี้จริง ดูแล้วอึ้ง แชร์ก่อนโดนลบ", "ข้อมูลบิดเบือน"),
]

print(f"มีข้อความทั้งหมด {len(SAMPLE_MESSAGES)} ข้อความ")
for text, answer in SAMPLE_MESSAGES[:3]:
    print(f"  [{answer}] {text[:50]}...")

### ด่านที่ 1 — คัดกรองด้วยโมเดลถูก (Haiku)

สองอย่างที่ทำให้โค้ดนี้ใช้ได้จริง ไม่ใช่แค่ของเล่น

**`format`** บังคับให้คำตอบเป็น JSON ตามโครงที่เรากำหนด ไม่ใช่ข้อความอิสระ
โปรแกรมเอาไปใช้ต่อได้ทันที ไม่ต้องมานั่งแกะข้อความ

**`effort: low`** บอกโมเดลว่างานนี้ไม่ต้องคิดลึก ประหยัดทั้งเงินและเวลา

In [ ]:
import json

TRIAGE_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "ประเภท": {"type": "string", "enum": ["สแกม", "ปกติ", "ข้อมูลบิดเบือน"]},
            "ความมั่นใจ": {"type": "number"},
            "สัญญาณอันตราย": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["ประเภท", "ความมั่นใจ", "สัญญาณอันตราย"],
        "additionalProperties": False,
    },
}

TRIAGE_PROMPT = """คุณคือผู้ช่วยคัดกรองข้อความหลอกลวงสำหรับคนไทย
อ่านข้อความที่ได้รับ แล้วจัดเป็นหนึ่งใน 3 ประเภท

- สแกม = พยายามหลอกเอาเงิน ข้อมูลส่วนตัว หรือให้กดลิงก์อันตราย
- ปกติ = ข้อความทั่วไปที่ไม่มีเจตนาหลอก
- ข้อมูลบิดเบือน = ข้อมูลเท็จหรือบิดเบือนที่ออกแบบมาให้คนแชร์ต่อ

ความมั่นใจ ให้เป็นเลข 0 ถึง 1
สัญญาณอันตราย ให้ระบุสิ่งที่ทำให้คุณตัดสินแบบนั้น เป็นวลีสั้น ๆ ภาษาไทย"""


def triage(text):
    """ส่งข้อความ 1 ข้อความให้ด่านคัดกรอง แล้วคืนผลเป็น dict"""
    r = client.messages.create(
        model=MODEL_FAST,
        max_tokens=2000,
        system=TRIAGE_PROMPT,
        messages=[{"role": "user", "content": text}],
        output_config={"effort": "low", "format": TRIAGE_SCHEMA},
    )
    out = json.loads(next(b.text for b in r.content if b.type == "text"))
    out["_tokens"] = (r.usage.input_tokens, r.usage.output_tokens)
    return out


# ลองกับข้อความแรก
result = triage(SAMPLE_MESSAGES[0][0])
print(json.dumps(result, ensure_ascii=False, indent=2))

### รันทั้งชุด

หมายเหตุสำหรับคนที่เคยใช้ Anthropic API มาก่อน: บน Foundry **ไม่มี Message Batches**
เราจึงวนลูปทีละข้อความ

In [ ]:
results = []
for text, answer in SAMPLE_MESSAGES:
    out = triage(text)
    results.append((text, answer, out))
    mark = "✅" if out["ประเภท"] == answer else "❌"
    print(f'{mark} ทาย={out["ประเภท"]:<14} เฉลย={answer:<14} {text[:38]}...')

n_correct = sum(1 for _, a, o in results if o["ประเภท"] == a)
print(f"\nความแม่น {n_correct}/{len(results)} = {n_correct / len(results) * 100:.0f}%")

---
## ขั้น 2 — ทำให้เป็นของทีมคุณ

ตรงนี้คือหัวใจ: **คำสั่งที่เขียนดี มีผลมากกว่าโมเดลที่แพงกว่า**

ลองทำทีละอย่าง แล้วรันเซลล์ `results` ข้างบนซ้ำ ดูว่าเลขความแม่นขยับไหม

1. เพิ่มกฎใน `TRIAGE_PROMPT` เช่น *"ข้อความที่เร่งให้ตัดสินใจภายในเวลาจำกัด ให้ถือเป็นสัญญาณอันตราย"*
2. เพิ่มประเภทใหม่ใน `enum` เช่น `"น่าสงสัย"` สำหรับเคสที่ไม่ชัด
3. เอาข้อความสแกมที่ทีมคุณเคยได้รับจริงมาใส่ข้างล่างนี้

In [ ]:
my_messages = [
    "เอาข้อความที่เคยได้รับจริงมาวางตรงนี้",
    "วางได้หลายอัน",
]

for text in my_messages:
    out = triage(text)
    print(f'{out["ประเภท"]:<14} (มั่นใจ {out["ความมั่นใจ"]:.2f})  {text[:45]}')
    print(f'   สัญญาณ: {", ".join(out["สัญญาณอันตราย"])}\n')

---
## ขั้น 3 (สายโค้ด) — ด่านที่ 2 ด้วยโมเดลฉลาด

ด่านแรกบอกว่า "น่าจะสแกม" แต่คนที่ได้รับข้อความอยากรู้ว่า **แล้วยังไงต่อ**

ด่านที่ 2 ใช้ Opus อธิบายเหตุผล และดึงของที่เอาไปแจ้งความหรือบล็อกได้จริง
แต่เรียกเฉพาะเคสที่ด่านแรกบอกว่าเป็นสแกม — **ไม่ใช่ทุกข้อความ**
นี่คือจุดที่ประหยัดเงินได้มากที่สุด

In [ ]:
DETAIL_SCHEMA = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "อธิบายให้เหยื่อเข้าใจ": {"type": "string"},
            "ลิงก์": {"type": "array", "items": {"type": "string"}},
            "เบอร์โทร": {"type": "array", "items": {"type": "string"}},
            "เลขบัญชี": {"type": "array", "items": {"type": "string"}},
            "ควรทำอย่างไรต่อ": {"type": "array", "items": {"type": "string"}},
        },
        "required": [
            "อธิบายให้เหยื่อเข้าใจ", "ลิงก์", "เบอร์โทร", "เลขบัญชี", "ควรทำอย่างไรต่อ",
        ],
        "additionalProperties": False,
    },
}

DETAIL_PROMPT = (
    "คุณคือผู้เชี่ยวชาญด้านอาชญากรรมไซเบอร์ "
    "อธิบายด้วยภาษาที่คนทั่วไปเข้าใจ ไม่ใช้ศัพท์เทคนิค "
    "ถ้าไม่พบข้อมูลประเภทไหน ให้ส่งเป็น list ว่าง"
)


def analyze_detail(text):
    r = client.messages.create(
        model=MODEL_SMART,
        max_tokens=4000,
        system=DETAIL_PROMPT,
        messages=[{"role": "user", "content": text}],
        output_config={"effort": "medium", "format": DETAIL_SCHEMA},
    )
    return json.loads(next(b.text for b in r.content if b.type == "text"))


scam_cases = [t for t, _, o in results if o["ประเภท"] == "สแกม"]
print(f"ด่านแรกคัดมา {len(scam_cases)} จาก {len(results)} ข้อความ "
      f"— ประหยัดการเรียกโมเดลแพงไป {len(results) - len(scam_cases)} ครั้ง\n")

detail = analyze_detail(scam_cases[0])
print(json.dumps(detail, ensure_ascii=False, indent=2))

### ต่อยอด: ให้มันเรียกเครื่องมือเองได้

จนถึงตอนนี้โมเดลตัดสินจากตัวข้อความอย่างเดียว ของจริงต้องเช็กกับฐานข้อมูลได้ด้วย

`strict: True` คือตัวที่รับประกันว่าโมเดลจะส่ง argument มาตรงตามโครงที่เรากำหนดเสมอ
ไม่ส่งอะไรแปลก ๆ มาให้โปรแกรมพัง

> **หมายเหตุสำคัญ** บน Claude Opus 5.5 **บังคับให้เรียก tool ไม่ได้แล้ว**
> (`tool_choice` แบบบังคับจะ error) วิธีที่ถูกคือเขียนในคำสั่งให้ชัดว่าเมื่อไหร่ควรเรียก

In [ ]:
# ฐานข้อมูลปลอมไว้สอน — ของจริงตรงนี้คือการต่อ API ของหน่วยงานหรือ blacklist กลาง
BLOCKLIST = {"th-parcel-pay.example", "bank-verify-th.example", "power-bill-th.example"}

TOOLS = [{
    "name": "check_domain",
    "description": "ตรวจว่าโดเมนนี้อยู่ในบัญชีดำหรือไม่ ใช้เมื่อพบลิงก์ในข้อความ",
    "strict": True,
    "input_schema": {
        "type": "object",
        "properties": {"domain": {"type": "string"}},
        "required": ["domain"],
        "additionalProperties": False,
    },
}]

test_message = SAMPLE_MESSAGES[0][0]

r = client.messages.create(
    model=MODEL_SMART,
    max_tokens=4000,
    system="ถ้าข้อความมีลิงก์ ให้เรียกเครื่องมือ check_domain ก่อนตอบเสมอ",
    messages=[{"role": "user", "content": test_message}],
    tools=TOOLS,
)

for b in r.content:
    if b.type == "tool_use":
        domain = b.input["domain"]
        listed = domain in BLOCKLIST
        print(f"โมเดลขอเช็ก: {domain}  →  {'🚨 อยู่ในบัญชีดำ' if listed else 'ไม่พบในบัญชีดำ'}")
    elif b.type == "text" and b.text.strip():
        print(b.text)

print("\n(ขั้นต่อไปที่ทีมคุณทำเองได้: ส่งผล tool_result กลับเข้าไป แล้วให้โมเดลสรุปอีกรอบ)")

---
## ขั้น 3 (สายไม่เขียนโค้ด)

ไม่ต้องแตะโค้ดก็ทำส่วนที่สำคัญที่สุดของ proposal ได้
นั่นคือ **ออกแบบว่าของนี้ไปถึงมือคนใช้ยังไง**

เปิดผลลัพธ์ JSON ที่ได้จากขั้น 1-2 แล้วตอบ 4 ข้อนี้ลงในเอกสารของทีม

1. **คนใช้คือใคร** ผู้สูงอายุ? นักศึกษา? พนักงานธนาคาร? — เลือกกลุ่มเดียว
2. **เขาเจอของเราตอนไหน** ก่อนกดลิงก์? ตอนกำลังจะโอน? หลังโดนหลอกแล้ว?
3. **มาทางไหน** LINE bot? แอป? ส่วนขยายเบราว์เซอร์? เบอร์ให้ส่งต่อข้อความมาเช็ก?
4. **เห็นอะไรบนจอ** วาดหน้าจอเดียวพอ แล้วเอา `ประเภท` `ความมั่นใจ` `สัญญาณอันตราย` ไปวางบนหน้าจอนั้น

> ทีมที่ชนะ hackathon มักไม่ใช่ทีมที่โมเดลแม่นที่สุด
> แต่เป็นทีมที่ตอบ 4 ข้อนี้ได้ชัดที่สุด

---
## วัดให้ได้ว่าแม่นแค่ไหน

คำถามแรกที่กรรมการจะถามคือ **"รู้ได้ยังไงว่ามันแม่น"**
ทีมที่ตอบไม่ได้เสียคะแนนตรงนี้ทุกครั้ง

คำตอบไม่ใช่ความรู้สึก แต่คือชุดทดสอบที่คุณทำเอง

In [ ]:
wrong = [(t, a, o["ประเภท"]) for t, a, o in results if o["ประเภท"] != a]
n_correct = len(results) - len(wrong)

print(f"ความแม่นรวม: {n_correct}/{len(results)} = {n_correct / len(results) * 100:.0f}%\n")

print("แม่นแยกตามประเภท")
for label in ["สแกม", "ปกติ", "ข้อมูลบิดเบือน"]:
    group = [(a, o) for _, a, o in results if a == label]
    hit = sum(1 for a, o in group if o["ประเภท"] == a)
    print(f"  {label:<14} {hit}/{len(group)}")

print("\nเคสที่ทายผิด — อ่านให้ครบ นี่คือของที่เอาไปปรับคำสั่ง")
if not wrong:
    print("  (รอบนี้ไม่มี — ลองหาข้อความที่ยากกว่านี้มาใส่)")
for t, a, pred in wrong:
    print(f"  เฉลย {a} แต่ทาย {pred}: {t[:55]}...")

print("\n--- ข้อควรระวังที่ต้องเขียนลง proposal ---")
print("ทายผิดแบบ 'สแกมแต่บอกว่าปกติ' อันตรายกว่า 'ปกติแต่บอกว่าสแกม' มาก")
print("เพราะแบบแรกคือปล่อยให้เหยื่อโดนหลอก ส่วนแบบหลังแค่สร้างความรำคาญ")
print("ตอนออกแบบ ให้เลือกว่าจะยอมพลาดด้านไหน แล้วอธิบายเหตุผลให้กรรมการฟัง")

### ค่าใช้จ่ายจริง

ตัวเลขนี้ต้องอยู่ใน proposal ของคุณ
เพราะ **โซลูชันที่แพงเกินกว่าจะใช้จริง คือโซลูชันที่ไม่มีใครได้ใช้**

In [ ]:
PRICE = {  # ดอลลาร์ ต่อ 1 ล้านโทเคน
    MODEL_FAST:  {"in": 0.10, "out": 0.50},
    MODEL_SMART: {"in": 4.00, "out": 20.00},
}
THB_PER_USD = 33  # ปรับให้ตรงวันที่ใช้งานจริง

tok_in  = sum(o["_tokens"][0] for _, _, o in results)
tok_out = sum(o["_tokens"][1] for _, _, o in results)

cost_usd = (tok_in / 1e6) * PRICE[MODEL_FAST]["in"] + (tok_out / 1e6) * PRICE[MODEL_FAST]["out"]
per_msg_thb = cost_usd / len(results) * THB_PER_USD

print(f"ด่านคัดกรอง {len(results)} ข้อความ ใช้ {tok_in:,} โทเคนเข้า / {tok_out:,} โทเคนออก")
print(f"คิดเป็น {per_msg_thb:.4f} บาทต่อข้อความ\n")
print(f"ถ้ามีคนใช้ 10,000 ข้อความต่อเดือน       = {per_msg_thb * 10000:,.0f} บาท/เดือน")
print(f"ถ้าใช้โมเดลแพงกับทุกข้อความแทน (ราว 40 เท่า) = {per_msg_thb * 10000 * 40:,.0f} บาท/เดือน")
print("\nนี่คือเหตุผลที่เราแยกสองด่าน")

---
## เคสจริง: อีเมลที่ไม่ขอเงินสักบาท

ข้อความที่ผ่านมาเป็นสแกมที่ดูออกง่าย เพราะมันขอเงินหรือขอให้กดลิงก์

ของจริงที่อันตรายกว่าคือข้อความที่ **ไม่ขออะไรที่ดูน่าสงสัยเลย** ตัวอย่างข้างล่างเป็นอีเมลที่
บริษัทแห่งหนึ่งได้รับจริง (ลบชื่อคน ชื่อบริษัท และเบอร์โทรออกแล้ว เหลือแต่โครงสร้าง)

ลองอ่านเองก่อนว่าคุณคิดว่าเป็นอะไร แล้วค่อยรันเซลล์ถัดไป

In [ ]:
อีเมลธุรกิจ = """สวัสดีค่ะ ทีมงาน [บริษัทของคุณ]

[ชื่อผู้ส่ง] จาก [ชื่อเอเจนซี] นะคะ เราเป็นเอเจนซีการตลาดที่ทำงานกับแบรนด์เทคโนโลยี
และกำลังมองหาพาร์ตเนอร์ในไทยเพื่อร่วมทำกิจกรรมด้าน AI, Workstation และ Server ค่ะ

อยากสอบถามว่า [บริษัทของคุณ] รับจัด Workstation ตามความต้องการของลูกค้า
รวมถึงอัปเกรดอุปกรณ์จัดเก็บข้อมูลจากแบรนด์อื่นด้วยไหมคะ
และพอจะแนะนำผู้ที่ดูแลเรื่อง Workstation หรือการร่วมมือทางธุรกิจให้พูดคุยด้วยได้หรือเปล่าคะ

ถ้าสะดวก รบกวนขอ LINE หรือ WhatsApp ของผู้ดูแลด้วยนะคะ
ทางเราอยากติดต่อแนะนำตัวและพูดคุยรายละเอียดเพิ่มเติมค่ะ

ขอบคุณมากค่ะ"""

ผล = triage(อีเมลธุรกิจ)
print("ด่านแรกตัดสินว่า:", ผล["ประเภท"], f"(มั่นใจ {ผล['ความมั่นใจ']:.2f})")
print("สัญญาณที่เห็น:", ", ".join(ผล["สัญญาณอันตราย"]) or "ไม่พบ")

### ทำไมด่านแรกถึงเอาไม่อยู่

อีเมลนี้ไม่มีลิงก์ ไม่มีไฟล์แนบ ไม่ขอรหัส ไม่เร่งเวลา ไม่ขอเงิน ภาษาสุภาพถูกต้อง
**ตัดสินจากตัวข้อความอย่างเดียวยังไงก็ไม่พอ** เพราะสิ่งที่ผิดปกติอยู่ข้างนอกข้อความ

สิ่งที่เขาขอจริง ๆ คือ **ชื่อและช่องทางส่วนตัวของคนที่มีอำนาจจัดซื้อ** ซึ่งเป็นก้าวแรกของ
การปลอมตัวเป็นคู่ค้าในภายหลัง ไม่ใช่การหลอกเอาเงินในอีเมลฉบับนี้

ลองเช็กข้อมูลภายนอกดู — คำถามเดียวที่ตอบง่ายที่สุดคือ **เอเจนซีเจ้านี้มีเว็บไซต์ไหม**

In [ ]:
import socket

def เว็บมีจริงไหม(domain):
    """เช็กว่าโดเมนชี้ไปที่เครื่องจริงหรือไม่ ใช้ได้เลยไม่ต้องลงไลบรารีเพิ่ม"""
    try:
        socket.gethostbyname(domain)
        return True
    except socket.gaierror:
        return False


# ลองกับโดเมนที่มีเว็บจริง เทียบกับโดเมนที่ไม่มี
for d in ["github.com", "example-agency-with-no-site.example"]:
    print(f"{d:45s} {'มีเว็บไซต์' if เว็บมีจริงไหม(d) else 'ไม่มีเว็บไซต์'}")

print()
print("--- สิ่งที่ตรวจพบในเคสจริง ---")
print("โดเมนผู้ส่ง จดทะเบียนมาแล้ว 7 ปี แต่ไม่มีเว็บไซต์เลย")
print("เมลโฮสต์อยู่คนละประเทศกับที่อ้างในอีเมล")
print("ไม่มี DMARC แปลว่าโดเมนถูกปลอมได้ง่าย")
print("กล่องเมลเป็นชื่อแคมเปญรายประเทศ ไม่ใช่ชื่อคนส่ง")
print()
print("เอเจนซีการตลาดที่ไม่มีเว็บไซต์ของตัวเอง ขัดกับธุรกิจที่อ้างโดยตรง")

### บทเรียนที่เอาไปใช้ต่อ

| เรื่อง | สแกมแบบในชุดข้อมูล | เคสจริงข้างบน |
|---|---|---|
| ขออะไร | เงิน หรือให้กดลิงก์ | ชื่อและช่องทางส่วนตัวของคน |
| ตัดสินจากข้อความได้ไหม | ได้ | ไม่ได้ ต้องเช็กข้อมูลภายนอก |
| ความเสียหาย | เกิดทันที | เกิดทีหลัง ตอนถูกปลอมตัวเป็นคู่ค้า |

**ถ้าทีมคุณจะทำเครื่องมือให้องค์กรใช้** อย่าออกแบบให้ตอบแค่ว่าใช่หรือไม่ใช่สแกม
ให้มีสถานะ **น่าสงสัย** ไว้ด้วย พร้อมบอกว่าต้องไปตรวจอะไรเพิ่ม
เพราะเคสที่สร้างความเสียหายมากที่สุด มักเป็นเคสที่อ่านแล้วดูปกติที่สุด

---
## คำถามถัดไป: ใครเป็นคนเขียนข้อความนี้

วิธีจับสแกมแบบเดิมคือดูคำผิดและภาษาแปลก ๆ **วิธีนั้นใช้ไม่ได้แล้ว**
เพราะคนร้ายใช้ AI แปลและเกลาภาษาได้ดีกว่าคนจริงหลายคน

คำถามที่มีประโยชน์กว่าจึงไม่ใช่ "ภาษาดีไหม" แต่เป็น **"ภาษาดีแบบที่คนในวงการนี้พูดจริงหรือเปล่า"**

ลองให้โมเดลประเมินผู้เขียนของอีเมลฉบับเดิมดู

In [ ]:
โครงผู้เขียน = {
    "type": "json_schema",
    "schema": {
        "type": "object",
        "properties": {
            "ผู้เขียน": {
                "type": "string",
                "enum": ["คนไทยเขียนเอง", "ชาวต่างชาติเขียนไทยเอง", "แปลด้วยเครื่องหรือเขียนด้วย AI"],
            },
            "ความมั่นใจ": {"type": "number"},
            "สัญญาณทางภาษา": {"type": "array", "items": {"type": "string"}},
            "คำที่ไม่เป็นธรรมชาติ": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["ผู้เขียน", "ความมั่นใจ", "สัญญาณทางภาษา", "คำที่ไม่เป็นธรรมชาติ"],
        "additionalProperties": False,
    },
}

คำสั่งผู้เขียน = """คุณคือผู้เชี่ยวชาญภาษาไทย อ่านข้อความแล้วประเมินว่าใครน่าจะเป็นผู้เขียน

สิ่งที่ต้องดู
- โครงสร้างประโยคที่แปลตรงมาจากภาษาอังกฤษ
- คำศัพท์ที่แปลตามพจนานุกรม แทนคำที่คนในวงการนั้นใช้จริง
- การใช้ ค่ะ ครับ นะคะ ถูกตำแหน่งหรือไม่ และสม่ำเสมอจนผิดธรรมชาติหรือไม่
- การไม่มีสรรพนามบุรุษที่หนึ่งเลยทั้งฉบับ
- สำนวนธุรกิจไทยที่ควรมีแต่กลับไม่มี
- การใช้เครื่องหมายวรรคตอนแบบภาษาอังกฤษ

คำที่ไม่เป็นธรรมชาติ ให้ยกคำหรือวลีจากข้อความมาตรง ๆ"""


def วิเคราะห์ผู้เขียน(text):
    r = client.messages.create(
        model=MODEL_SMART,
        max_tokens=4000,
        system=คำสั่งผู้เขียน,
        messages=[{"role": "user", "content": text}],
        output_config={"effort": "medium", "format": โครงผู้เขียน},
    )
    return json.loads(next(b.text for b in r.content if b.type == "text"))


ผลผู้เขียน = วิเคราะห์ผู้เขียน(อีเมลธุรกิจ)
print("ผู้เขียน :", ผลผู้เขียน["ผู้เขียน"], f"(มั่นใจ {ผลผู้เขียน['ความมั่นใจ']:.2f})")
print()
print("สัญญาณทางภาษา")
for x in ผลผู้เขียน["สัญญาณทางภาษา"]:
    print("  -", x)
print()
print("คำที่ไม่เป็นธรรมชาติ")
for x in ผลผู้เขียน["คำที่ไม่เป็นธรรมชาติ"]:
    print("  -", x)

### ตัวชี้ที่ใช้ได้จริงกับข้อความภาษาไทย

| สิ่งที่เห็น | ทำไมถึงน่าสงสัย |
|---|---|
| ศัพท์แปลตามพจนานุกรม เช่น อุปกรณ์จัดเก็บข้อมูล | คนในวงการพูดว่า สตอเรจ ฮาร์ดดิสก์ หรือ SSD |
| ชื่อคนติดกับคำว่า จาก โดยไม่มีสรรพนาม | เป็นโครงสร้าง Name from Company ที่ยกมาทั้งท่อน |
| ทั้งฉบับไม่มี ดิฉัน หรือ ผม เลย | อีเมลธุรกิจไทยมีสรรพนามอย่างน้อยหนึ่งครั้งเสมอ |
| ค่ะ ครับ ถูกตำแหน่งแต่สม่ำเสมอเท่ากันหมด | คนเขียนจริงจะแกว่งมากกว่านี้ |
| ไม่มีคำผิดเลยสักคำ | อีเมลขายของจากคนจริงมักมีจุดที่หลุดอย่างน้อยหนึ่งจุด |
| ไม่มีสำนวนธุรกิจไทย เช่น เบื้องต้น ทั้งนี้ | แม่แบบที่แปลมาจะไม่มีสำนวนเฉพาะถิ่น |
| ใช้จุลภาคคั่นรายการแบบอังกฤษ | ไทยใช้เว้นวรรคเปล่า |

### ข้อควรระวังที่สำคัญที่สุดของหัวข้อนี้

**เขียนด้วย AI ไม่ได้แปลว่าโกง** ทุกวันนี้บริษัทจำนวนมากใช้ AI แปลอีเมลหาลูกค้าต่างประเทศเป็นเรื่องปกติ
ถ้าทีมคุณทำเครื่องมือที่ตัดสินว่าเป็นสแกมเพราะข้อความเขียนด้วย AI คุณจะกล่าวหาคนบริสุทธิ์จำนวนมาก

ให้ใช้ผลนี้เป็น **หนึ่งในหลายสัญญาณ** เท่านั้น เคสในบทนี้น่าสงสัยเพราะสามอย่างมาพร้อมกัน
คือภาษาที่ไม่ได้มาจากคนไทย โดเมนที่ไม่มีเว็บไซต์ และการขอช่องทางส่วนตัวของผู้มีอำนาจจัดซื้อ
ถ้ามีแค่อย่างแรกอย่างเดียว ยังสรุปอะไรไม่ได้เลย

---
## จากต้นแบบสู่การใช้งานจริง — ปกปิดข้อมูลส่วนบุคคลก่อนบันทึก

เครื่องมือของคุณรับ **ข้อความของคนอื่น** เข้ามา ซึ่งมักมีเบอร์โทร เลขบัญชี หรือชื่อติดมาด้วย

ถ้าเก็บ log ดิบไว้ตรง ๆ วันหนึ่งที่ระบบถูกเจาะ สิ่งที่หลุดออกไปคือข้อมูลของคนที่ไว้ใจคุณ
วิธีที่ถูกคือ **ปกปิดก่อนบันทึก ไม่ใช่บันทึกก่อนแล้วค่อยลบ**

ลองรันเซลล์ข้างล่าง แล้วแก้ให้ครอบคลุมรูปแบบที่ทีมคุณเจอจริง

In [ ]:
import re

def ปกปิด(text):
    """ปกปิดข้อมูลส่วนบุคคลก่อนนำไปบันทึกหรือส่งต่อ"""
    text = re.sub(r"\b0\d{1,2}[-\s]?\d{3}[-\s]?\d{3,4}\b", "[เบอร์โทร]", text)
    text = re.sub(r"\b\d{3}[-\s]?\d[-\s]?\d{5}[-\s]?\d\b", "[เลขบัญชี]", text)
    text = re.sub(r"\bhttps?://\S+|\b[\w.-]+\.(?:com|net|example)\S*", "[ลิงก์]", text)
    return text


for text, _ in SAMPLE_MESSAGES[:6]:
    before = text[:60]
    after = ปกปิด(text)[:60]
    if before != after:
        print("เดิม :", before)
        print("ปกปิด:", after, "\n")

print("--- สิ่งที่ทีมคุณต้องตัดสินใจ และเขียนลงเอกสารของทีม ---")
print("1. เก็บข้อความดิบไว้นานแค่ไหน และเก็บไว้ทำไม")
print("2. ใครในทีมเข้าถึง log ได้บ้าง")
print("3. ถ้าผู้ใช้ขอให้ลบข้อมูลของเขา ระบบของคุณทำได้ไหม")

---
## ต่อจากวันนี้

### เส้นตายที่ต้องจำ — ส่ง Concept Proposal ภายใน 31 ธันวาคม 2569

สิ่งที่ควรหยิบจาก notebook นี้ไปใส่ใน proposal

- ปัญหาที่เลือก และคนใช้คือใคร (จาก 4 ข้อในขั้น 3)
- ภาพสถาปัตยกรรม: ข้อความเข้า → ด่านคัดกรองถูก → ด่านวิเคราะห์ละเอียด → ผลถึงผู้ใช้
- **ตัวเลขความแม่นจากชุดทดสอบของคุณเอง** พร้อมบอกตรง ๆ ว่าเคสไหนยังพลาด
- ค่าใช้จ่ายต่อข้อความ และต่อผู้ใช้ 10,000 ข้อความต่อเดือน
- ขอบเขตที่ทำเสร็จได้ใน hackathon 3 วัน — **เล็กแต่ทำงานได้จริง ดีกว่าใหญ่แต่ทำไม่เสร็จ**

### ข้อเตือนสุดท้าย
key ที่แจกวันนี้ถูกยกเลิกเย็นนี้
ถ้าจะทำต่อ คุยกับทีมงานเรื่องช่องทางเข้าถึงโมเดลสำหรับช่วงเขียน proposal

---
*Thailand Cyber & Digital Innovation Challenge · U.S. Embassy Bangkok · depa*